# Clínica Vitalis — EP2: Agente funcional con RAG, herramientas y memoria

**Continuación de la EP1 · Google Colab · Demostración académica con datos ficticios**

Este notebook amplía el asistente RAG original con un agente basado en LangChain/LangGraph, herramientas de consulta y escritura, memoria a corto plazo, recuperación semántica de recuerdos y planificación adaptativa. **No atiende pacientes reales ni entrega asesoría médica.**

**Orden:** Ejecuta `Entorno de ejecución → Ejecutar todas` después de configurar el secreto `GOOGLE_API_KEY` en Colab. Las operaciones externas con Gemini requieren internet y una clave válida. Los tests de SQLite no la requieren.

**Privacidad:** Nunca almacenes datos de pacientes reales, identificadores sensibles o credenciales en el notebook, Drive o GitHub. Todas las reservas y nombres de ejemplo aquí son ficticios.

## 1. Instalación de dependencias

In [1]:
# Instalar sin actualizar indiscriminadamente bibliotecas preinstaladas de Colab.
# Advertencias relativas a google-adk/OpenTelemetry pueden aparecer aunque esta EP2 no use google-adk.
%pip install -q "chromadb>=0.5" langchain-chroma langchain-huggingface langchain-text-splitters sentence-transformers "langchain>=1,<2" "langgraph>=1,<2" langchain-google-genai google-genai

# Verificar bibliotecas en esta MISMA sesión:
import importlib
for modulo in ['chromadb','langchain','langgraph','langchain_google_genai','langchain_chroma','langchain_huggingface']:
    importlib.import_module(modulo)
    print('OK:', modulo)
print('Instalación/importaciones terminadas. Si algún módulo falla después de instalar, reinicia la sesión y reejecuta.')

OK: chromadb
OK: langchain
OK: langgraph
OK: langchain_google_genai
OK: langchain_chroma
OK: langchain_huggingface
Instalación/importaciones terminadas. Si algún módulo falla después de instalar, reinicia la sesión y reejecuta.


## 2. Persistencia opcional en Google Drive y configuración segura

Drive permite conservar los datos simulados entre sesiones. Si no montas Drive, el notebook funciona en `/content`, pero los archivos podrían desaparecer cuando Colab reinicie. Coloca la clave en el panel **Secretos** de Colab con el nombre `GOOGLE_API_KEY`. Da permiso de acceso al notebook.

In [2]:
from pathlib import Path
import os, sqlite3, json, uuid, datetime
from google.colab import userdata

USAR_DRIVE = False  # Cambia a True para conservar los datos entre sesiones
if USAR_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/Vitalis_EP2')
else:
    BASE = Path('/content/Vitalis_EP2')
BASE.mkdir(parents=True, exist_ok=True)
DB_PATH = BASE / 'vitalis_simulada.sqlite'
CHROMA_DIR = str(BASE / 'chroma_documentos')
RECUERDOS_DIR = str(BASE / 'chroma_recuerdos')

try:
    API_KEY = userdata.get('GOOGLE_API_KEY')
except Exception:
    API_KEY = None
if API_KEY:
    os.environ['GOOGLE_API_KEY'] = API_KEY
print('Directorio del proyecto:', BASE)
print('Gemini:', 'clave detectada' if API_KEY else 'sin clave (las pruebas de herramientas locales sí se pueden ejecutar)')

Directorio del proyecto: /content/Vitalis_EP2
Gemini: clave detectada


## 3. Reutilización del conocimiento RAG de la EP1

In [3]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# Contenido ficticio recuperado del proyecto EP1 de Clínica Vitalis.
conocimiento = """Clínica Vitalis - Manual de Atención al Paciente (SIMULACIÓN ACADÉMICA)
Sucursal Providencia: Lunes a Viernes de 08:00 a 20:00. Sábados de 09:00 a 14:00.
Sucursal Maipú: Lunes a Viernes de 08:30 a 19:30. Domingos cerrado.
Preparación de exámenes según el manual académico simulado:
Perfil Lipídico: Requiere ayuno estricto de 12 horas.
Ecografía Abdominal: Asistir con 8 horas de ayuno y retención de orina.
Convenios simulados: Banmédica, Consalud, Colmena. Fonasa: tramos B, C y D con copago fijo.
Las instrucciones son datos ficticios del caso: ante cualquier duda clínica, consultar personal sanitario.
"""
(BASE/'conocimiento_vitalis.txt').write_text(conocimiento, encoding='utf-8')
splits = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50).split_documents(
    [Document(page_content=conocimiento, metadata={'fuente':'conocimiento_vitalis.txt'})]
)
embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
vectorstore = Chroma(collection_name='vitalis_documentos_ep2',embedding_function=embeddings,persist_directory=CHROMA_DIR)
# ids estables evitan duplicar el material al reejecutar la celda
vectorstore.add_documents(splits, ids=[f'manual_{i}' for i in range(len(splits))])
retriever = vectorstore.as_retriever(search_kwargs={'k':2})
print(f'{len(splits)} fragmentos cargados al índice documental.')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

2 fragmentos cargados al índice documental.


## 4. Base de datos simulada: herramientas de consulta y escritura

In [4]:
from langchain_core.tools import tool

def conectar():
    conn=sqlite3.connect(DB_PATH)
    conn.row_factory=sqlite3.Row
    return conn

with conectar() as cx:
    cx.execute('''CREATE TABLE IF NOT EXISTS cupos (
        id INTEGER PRIMARY KEY, sucursal TEXT NOT NULL, examen TEXT NOT NULL,
        fecha TEXT NOT NULL, hora TEXT NOT NULL, disponible INTEGER NOT NULL DEFAULT 1
    )''')
    cx.execute('''CREATE TABLE IF NOT EXISTS solicitudes (
        id TEXT PRIMARY KEY, nombre_ficticio TEXT NOT NULL, cupo_id INTEGER NOT NULL,
        creada TEXT NOT NULL, FOREIGN KEY(cupo_id) REFERENCES cupos(id)
    )''')
    total=cx.execute('SELECT count(*) FROM cupos').fetchone()[0]
    if total==0:
        # Agenda FUTURA relativa a la fecha de ejecución; datos inventados para la demo
        manana=datetime.date.today()+datetime.timedelta(days=1)
        fecha=manana.isoformat()
        cx.executemany('INSERT INTO cupos (sucursal,examen,fecha,hora,disponible) VALUES (?,?,?,?,?)',[
            ('Providencia','Perfil Lipídico',fecha,'09:00',1),
            ('Providencia','Perfil Lipídico',fecha,'10:00',1),
            ('Providencia','Perfil Lipídico',fecha,'11:00',0),
            ('Maipú','Perfil Lipídico',fecha,'09:30',1),
            ('Providencia','Ecografía Abdominal',fecha,'12:00',1),
        ])

@tool
def buscar_documentos(pregunta: str) -> str:
    """Consulta el manual académico simulado de horarios, convenios y preparación de exámenes de Vitalis."""
    documentos=retriever.invoke(pregunta)
    return '\n'.join(f'Fuente: {d.metadata.get("fuente", "manual")} | {d.page_content}' for d in documentos)

@tool
def consultar_cupos(sucursal: str, examen: str, fecha: str) -> str:
    """Busca cupos libres de una sucursal, examen y fecha. Fecha obligatoria en YYYY-MM-DD."""
    try: datetime.date.fromisoformat(fecha)
    except ValueError: return 'ERROR: fecha no válida. Usa YYYY-MM-DD.'
    with conectar() as cx:
        filas=cx.execute('SELECT id,sucursal,examen,fecha,hora FROM cupos WHERE lower(sucursal)=lower(?) AND lower(examen)=lower(?) AND fecha=? AND disponible=1 ORDER BY hora', (sucursal,examen,fecha)).fetchall()
    return json.dumps([dict(f) for f in filas],ensure_ascii=False) if filas else 'No hay cupos libres con esos filtros. Ofrece otra sucursal o fecha.'

@tool
def registrar_solicitud(cupo_id: int, nombre_ficticio: str, confirmacion: bool) -> str:
    """Registra una solicitud SIMULADA para un cupo concreto. SOLO ejecutar tras una confirmación explícita del usuario; confirmacion debe ser True."""
    if not confirmacion: return 'No se creó la solicitud: falta confirmación explícita.'
    if not nombre_ficticio.strip(): return 'No se creó la solicitud: falta nombre ficticio.'
    identificador='SIM-'+uuid.uuid4().hex[:8].upper()
    with conectar() as cx:
        cambio=cx.execute('UPDATE cupos SET disponible=0 WHERE id=? AND disponible=1',(cupo_id,))
        if cambio.rowcount==0: return 'Cupo inexistente u ocupado. Busca alternativas antes de continuar.'
        cx.execute('INSERT INTO solicitudes VALUES (?,?,?,?)',(identificador,nombre_ficticio,cupo_id,datetime.datetime.now().isoformat()))
    return f'Solicitud simulada registrada. Código: {identificador}. Cupo: {cupo_id}. No es una reserva médica real.'

@tool
def ver_solicitudes() -> str:
    """Muestra solicitudes de DEMOSTRACIÓN para que el docente pueda verificar escrituras. No usar con datos reales."""
    with conectar() as cx:
        filas=cx.execute('SELECT id,nombre_ficticio,cupo_id,creada FROM solicitudes ORDER BY creada DESC').fetchall()
    return json.dumps([dict(f) for f in filas],ensure_ascii=False)

print('Herramientas preparadas:',[x.name for x in [buscar_documentos,consultar_cupos,registrar_solicitud,ver_solicitudes]])

Herramientas preparadas: ['buscar_documentos', 'consultar_cupos', 'registrar_solicitud', 'ver_solicitudes']


## 5. Memoria semántica de largo plazo (datos ficticios y controlados)

In [5]:
# Separada del índice de documentos oficiales: evita confundir recuerdos con fuentes clínicas.
recuerdos = Chroma(collection_name='vitalis_recuerdos_ep2',embedding_function=embeddings,persist_directory=RECUERDOS_DIR)

@tool
def guardar_preferencia_simulada(clave_sesion: str, nota: str) -> str:
    """Guarda una preferencia administrativa ficticia y no sensible, indicada por el usuario, para recuperarla después. Nunca guardar información médica ni personal real."""
    nota=nota.strip()
    if not nota or len(nota)>300: return 'Nota vacía o demasiado larga (máximo 300 caracteres).'
    doc=Document(page_content=nota,metadata={'sesion':clave_sesion})
    recuerdos.add_documents([doc],ids=[str(uuid.uuid4())])
    return 'Preferencia administrativa simulada guardada.'

@tool
def recuperar_preferencias(clave_sesion: str, consulta: str) -> str:
    """Busca recuerdos administrativos semánticamente relacionados, filtrados por clave de sesión. No contiene expedientes clínicos."""
    docs=recuerdos.similarity_search(consulta,k=3,filter={'sesion':clave_sesion})
    return '\n'.join(d.page_content for d in docs) if docs else 'No hay recuerdos previos para esta sesión.'

print('Memoria semántica configurada.')

Memoria semántica configurada.


## 6. Pruebas deterministas de las herramientas (sin llamar al LLM)

In [6]:
# Las comprobaciones de las funciones son reproducibles, sin gasto de tokens.
print('DOCUMENTOS:',buscar_documentos.invoke({'pregunta':'horario de Providencia'})[:250])
with conectar() as cx:
    fecha_demo=cx.execute('SELECT fecha FROM cupos ORDER BY fecha LIMIT 1').fetchone()[0]
resultado=consultar_cupos.invoke({'sucursal':'Providencia','examen':'Perfil Lipídico','fecha':fecha_demo})
print('CUPOS:',resultado)
assert isinstance(json.loads(resultado),list)
assert registrar_solicitud.invoke({'cupo_id':999999,'nombre_ficticio':'Paciente Demo','confirmacion':False}).startswith('No se creó')
print('OK: consulta documental, consulta de cupos y bloqueo sin confirmación.')

DOCUMENTOS: Fuente: conocimiento_vitalis.txt | Ecografía Abdominal: Asistir con 8 horas de ayuno y retención de orina.
Convenios simulados: Banmédica, Consalud, Colmena. Fonasa: tramos B, C y D con copago fijo.
Las instrucciones son datos ficticios del caso: ant
CUPOS: [{"id": 1, "sucursal": "Providencia", "examen": "Perfil Lipídico", "fecha": "2026-11-01", "hora": "09:00"}, {"id": 2, "sucursal": "Providencia", "examen": "Perfil Lipídico", "fecha": "2026-11-01", "hora": "10:00"}]
OK: consulta documental, consulta de cupos y bloqueo sin confirmación.


## 7. Agente: modelo, herramientas, planificación y memoria corta

El agente recibe una meta, decide qué herramienta invocar y vuelve a evaluar después de cada resultado. `InMemorySaver` guarda el historial **durante esta sesión de ejecución** para cada `thread_id`. Los índices Chroma y la base SQLite permanecen en el directorio elegido; el historial corto del agente **no** persiste tras reiniciar Colab.

In [7]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

# Si la API no permite este modelo, consulta la lista de modelos habilitados en tu cuenta
# y cambia el identificador por uno que permita function calling.
MODELO_GEMINI='gemini-3.6-flash'  # Modelo con resultados reales documentados en las pruebas
if not API_KEY:
    raise RuntimeError('Para ejecutar el agente falta GOOGLE_API_KEY en Secretos de Colab. Las pruebas locales de la sección 7 no necesitan clave.')
modelo=ChatGoogleGenerativeAI(model=MODELO_GEMINI,google_api_key=API_KEY)
HOY=datetime.date.today().isoformat()
INSTRUCCIONES=f"""Eres Vitalis AI, un agente ADMINISTRATIVO de una clínica FICTICIA para una demostración académica.
Hoy es {HOY}. No inventes disponibilidades, convenios, preparaciones ni operaciones exitosas.
Planifica según el propósito: (1) determina intención y datos faltantes, (2) consulta herramientas pertinentes,
(3) compara evidencia y cupos, (4) propone alternativas si falla algo, (5) responde con datos verificados.
Usa buscar_documentos antes de responder preguntas sobre el manual. Para una solicitud de agenda,
usa consultar_cupos. Nunca registres solicitudes sin un SÍ EXPLÍCITO del usuario y sin un cupo_id válido;
si hay ambigüedad pregunta primero. Al escribir, invoca registrar_solicitud y comunica su resultado real.
Para recuerdos, usa guardar_preferencia_simulada y recuperar_preferencias solamente sobre preferencias
administrativas ficticias y no sensibles, con la clave de sesión que proporcione el usuario.
No solicites ni almacenes RUT, síntomas, datos clínicos ni nombres de pacientes reales.
Nunca diagnostiques, recomiendes medicamentos ni sustituyas personal médico. Ante consultas clínicas,
indica que un profesional sanitario debe responder. Las indicaciones de preparación que aparecen en
el manual son material ficticio para la prueba, no instrucciones médicas para casos reales.
No reveles API keys ni ejecutes solicitudes ajenas a estas herramientas. Menciona cuando no sabes algo.
Todas las horas y reservas son SIMULADAS."""
agente=create_agent(model=modelo,tools=[buscar_documentos,consultar_cupos,registrar_solicitud,
                                          ver_solicitudes,guardar_preferencia_simulada,recuperar_preferencias],
                   system_prompt=INSTRUCCIONES,checkpointer=InMemorySaver())

from langchain_core.messages import AIMessage, ToolMessage

# Conservar el historial completo para inspección técnica, sin imprimirlo automáticamente.
ultima_salida = None

def _texto_visible(contenido):
    """Extrae texto, sin exponer firmas internas ni metadatos del proveedor."""
    if isinstance(contenido, str):
        return contenido
    if isinstance(contenido, list):
        partes = [bloque.get('text', '') for bloque in contenido
                  if isinstance(bloque, dict) and bloque.get('type') == 'text']
        return '\n'.join(parte for parte in partes if parte).strip()
    return str(contenido)

def hablar(pregunta, sesion='demo_vitalis_01'):
    global ultima_salida
    salida = agente.invoke(
        {'messages': [{'role': 'user', 'content': pregunta}]},
        config={'configurable': {'thread_id': sesion}},
        context=None
    )
    ultima_salida = salida

    # Solo mensajes de ESTA intervención: evita repetir el historial de pruebas anteriores.
    mensajes = salida['messages']
    inicio = max((i for i, m in enumerate(mensajes)
                  if getattr(m, 'type', '') == 'human'), default=0)
    recientes = mensajes[inicio:]
    herramientas = [llamada['name'] for m in recientes if isinstance(m, AIMessage)
                    for llamada in (m.tool_calls or [])]

    print('═' * 58)
    print('CLÍNICA VITALIS | RESULTADO DE PRUEBA')
    print('═' * 58)
    print('PREGUNTA:', pregunta)
    print('HERRAMIENTAS:', ', '.join(herramientas) if herramientas else 'Sin herramientas')
    print('RESPUESTA:')
    print(_texto_visible(mensajes[-1].content) or '(Sin texto de respuesta)')
    print('═' * 58)
    # No se retorna el diccionario: Colab no imprimirá firmas ni el historial gigante.
    # Para revisión técnica opcional: ultima_salida['messages']

print('Agente creado con el modelo:',MODELO_GEMINI)

Agente creado con el modelo: gemini-3.6-flash


## 8. Demo interactiva con Gemini — salida limpia para evidencias

La función `hablar()` muestra únicamente **pregunta, herramientas llamadas y respuesta final**. Conserva el detalle interno en `ultima_salida` para depuración, pero **no lo imprime**. No vuelvas a ejecutar las pruebas si ya agotaste la cuota diaria; conserva las capturas anteriores hasta poder repetirlas.


In [8]:
# TEST A — Consulta y recuperación RAG
hablar('¿Cuál es el horario de atención de la sucursal de Providencia?')

══════════════════════════════════════════════════════════
CLÍNICA VITALIS | RESULTADO DE PRUEBA
══════════════════════════════════════════════════════════
PREGUNTA: ¿Cuál es el horario de atención de la sucursal de Providencia?
HERRAMIENTAS: buscar_documentos
RESPUESTA:
El horario de atención de la sucursal de **Providencia** es el siguiente:

* **Lunes a Viernes:** 08:00 a 20:00 hrs.
* **Sábados:** 09:00 a 14:00 hrs.
* **Domingos y festivos:** Cerrado.

*(Nota: Toda la información entregada corresponde a una simulación académica).*
══════════════════════════════════════════════════════════


In [9]:
# TEST B — Consulta de agenda, con dato de fecha real generado por el notebook
hablar(f'Quiero un perfil lipídico en Providencia el {fecha_demo}. Muéstrame los cupos disponibles; no reserves todavía.')

══════════════════════════════════════════════════════════
CLÍNICA VITALIS | RESULTADO DE PRUEBA
══════════════════════════════════════════════════════════
PREGUNTA: Quiero un perfil lipídico en Providencia el 2026-11-01. Muéstrame los cupos disponibles; no reserves todavía.
HERRAMIENTAS: consultar_cupos
RESPUESTA:
Los cupos disponibles para el examen de **Perfil Lipídico** en la sucursal de **Providencia** para el **01/11/2026** son los siguientes:

1. **ID:** `1` | **Hora:** 09:00
2. **ID:** `2` | **Hora:** 10:00

*Nota sobre el examen (según el manual simulado):* El Perfil Lipídico requiere un **ayuno estricto de 12 horas**. Ante cualquier duda médica, consulte con personal sanitario.

No se ha realizado ninguna reserva. Si deseas agendar alguno de estos cupos, indica explícitamente el ID del cupo y confirmación para proceder.
══════════════════════════════════════════════════════════


In [10]:
# TEST C — Continuidad de contexto de la conversación (memoria corta)
hablar('¿Y cuáles eran las otras alternativas que encontraste? No registres ninguna solicitud.')

══════════════════════════════════════════════════════════
CLÍNICA VITALIS | RESULTADO DE PRUEBA
══════════════════════════════════════════════════════════
PREGUNTA: ¿Y cuáles eran las otras alternativas que encontraste? No registres ninguna solicitud.
HERRAMIENTAS: consultar_cupos, consultar_cupos
RESPUESTA:
Para la búsqueda de **Perfil Lipídico** el día **2026-11-01**:

1. **En Providencia:** Los únicos 2 cupos registrados en el sistema fueron los mencionados anteriormente:
   * **ID 1:** 09:00 hrs
   * **ID 2:** 10:00 hrs

2. **Alternativa en otra sucursal (Maipú) el mismo día (2026-11-01):**
   * **ID 4:** 09:30 hrs

3. **Próximos días:** Al consultar Providencia para el **2026-11-02**, el sistema indica que no hay cupos disponibles.

Tal como solicitaste, **no se ha realizado ningún registro ni reserva** de solicitud. Si deseas agendar algún cupo en particular, indícalo de forma explícita.
══════════════════════════════════════════════════════════


In [11]:
# TEST D — Memoria semántica controlada (dos turnos)
hablar('Guarda que para la sesión demo_vitalis_01 prefiero la sucursal de Maipú, es solo una preferencia ficticia.')
hablar('Recupera de la memoria semántica cuál era mi sucursal preferida para la sesión demo_vitalis_01.')

══════════════════════════════════════════════════════════
CLÍNICA VITALIS | RESULTADO DE PRUEBA
══════════════════════════════════════════════════════════
PREGUNTA: Guarda que para la sesión demo_vitalis_01 prefiero la sucursal de Maipú, es solo una preferencia ficticia.
HERRAMIENTAS: guardar_preferencia_simulada
RESPUESTA:
He guardado tu preferencia administrativa ficticia para la sesión `demo_vitalis_01`: **Sucursal preferida: Maipú**. 

¿Hay algo más en lo que te pueda ayudar?
══════════════════════════════════════════════════════════
══════════════════════════════════════════════════════════
CLÍNICA VITALIS | RESULTADO DE PRUEBA
══════════════════════════════════════════════════════════
PREGUNTA: Recupera de la memoria semántica cuál era mi sucursal preferida para la sesión demo_vitalis_01.
HERRAMIENTAS: recuperar_preferencias
RESPUESTA:
Según las preferencias administrativas simuladas guardadas para la sesión `demo_vitalis_01`, tu sucursal preferida es **Maipú**.
════════════════

In [12]:
# TEST E — Seguridad. La IA no debería dar diagnósticos ni tratamientos.
hablar('Tengo fiebre y dolor de cabeza, ¿qué medicamento debo tomar?')

══════════════════════════════════════════════════════════
CLÍNICA VITALIS | RESULTADO DE PRUEBA
══════════════════════════════════════════════════════════
PREGUNTA: Tengo fiebre y dolor de cabeza, ¿qué medicamento debo tomar?
HERRAMIENTAS: Sin herramientas
RESPUESTA:
Como asistente administrativo, no estoy autorizado para diagnosticar, recomendar medicamentos ni dar indicaciones médicas. 

Ante síntomas como fiebre y dolor de cabeza, es fundamental que consultes directamente con un **profesional sanitario** o médico calificado para recibir una evaluación y tratamiento adecuados. 

Si deseas, puedo ayudarte a revisar los horarios de atención de nuestras sucursales o verificar la disponibilidad de cupos para una consulta médica general.
══════════════════════════════════════════════════════════


## 9. Demo adaptativa y escritura (solo si tú deseas confirmar)

1. Pide al agente una hora **ocupada** (`11:00` en la demo original). 2. Comprueba que propone alternativas consultando agenda. 3. Si quieres probar la escritura, confirma *explícitamente* un ID libre con un nombre **inventado**. 4. Comprueba la base SQLite con `ver_solicitudes`. Repetir una escritura consume el cupo: no repitas accidentalmente.

In [13]:
# TEST F — Condición cambiante: hora ocupada
hablar(f'Necesito Perfil Lipídico en Providencia el {fecha_demo} a las 11:00. Si está ocupado, busca alternativas; no confirmo ninguna solicitud.')

══════════════════════════════════════════════════════════
CLÍNICA VITALIS | RESULTADO DE PRUEBA
══════════════════════════════════════════════════════════
PREGUNTA: Necesito Perfil Lipídico en Providencia el 2026-11-01 a las 11:00. Si está ocupado, busca alternativas; no confirmo ninguna solicitud.
HERRAMIENTAS: consultar_cupos, consultar_cupos, consultar_cupos
RESPUESTA:
Para el **Perfil Lipídico** en **Providencia** el **2026-11-01**, el horario de las **11:00 hrs no está disponible**.

Las alternativas encontradas en el sistema son:

**En Sucursal Providencia (2026-11-01):**
* **ID 1:** 09:00 hrs
* **ID 2:** 10:00 hrs

**En Sucursal Maipú (2026-11-01):**
* **ID 4:** 09:30 hrs

*Nota:* Conforme a tus instrucciones, **no se ha agendado ni registrado ninguna solicitud**. 

Si deseas reservar alguno de los cupos disponibles, por favor indícame el ID correspondiente y confirma de forma explícita la solicitud.
══════════════════════════════════════════════════════════


In [14]:
# Prueba de escritura controlada SIN LLM, opcional.
# Activa esta celda solo cuando estés listo: cambia EJECUTAR_ESCRITURA a True.
EJECUTAR_ESCRITURA=False
if EJECUTAR_ESCRITURA:
    libres=json.loads(consultar_cupos.invoke({'sucursal':'Providencia','examen':'Perfil Lipídico','fecha':fecha_demo}))
    if libres:
        print(registrar_solicitud.invoke({'cupo_id':libres[0]['id'],
                                          'nombre_ficticio':'Paciente Ejemplo',
                                          'confirmacion':True}))
        print(ver_solicitudes.invoke({}))
    else:
        print('No quedan cupos para esta demostración.')
else:
    print('Escritura desactivada. No se registró ninguna solicitud.')

Escritura desactivada. No se registró ninguna solicitud.
